# ARMA — Univariate Benchmark (Turkey, Pipeline B)

**Model**: `ARIMA(2,0,0)` via `pmdarima`  
**Target**: `ngdprsaxdctrq` (quarterly log-diff real GDP, Turkey)  
**Variables**: Cat1 — target only (no external regressors)  
**Train**: 1995-Q2 to 2011-Q4 (67 eff. quarters) / **Test**: 2018-Q1 to 2025-Q4 (32 quarters)  
**Scaling**: No / **HP tuning**: No  
**Vintages**: m1 (Q-end −2m), m2 (Q-end −1m), m3 (Q-end month)  
**Note**: ARMA is purely univariate — vintages are identical (GDP pub_lag=2 masks current quarter at all three vintages).


In [1]:
import sys, os
import numpy as np
import pandas as pd
from pmdarima.arima import ARIMA
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.join(os.path.pardir, 'turkey_data'))
from turkey_helpers import (
    load_data, gen_lagged_data,
    PREDICTIONS_DIR, TARGET
)

SEED = 42
np.random.seed(SEED)

TRAIN_START = '1995-01-01'
TRAIN_END   = '2011-12-31'
TEST_START  = '2018-01-01'
TEST_END    = '2025-12-31'
VINTAGES    = {'m1': -2, 'm2': -1, 'm3': 0, 'post1': 1, 'post2': 2}

print('ARMA (Turkey) — univariate, target={}'.format(TARGET))


ARMA (Turkey) — univariate, target=ngdprsaxdctrq


In [2]:
monthly, _, metadata = load_data()

# Quarterly rows only — GDP is non-NaN at Q-end months
q_monthly = monthly[monthly['date'].dt.month.isin([3, 6, 9, 12])].copy()
print('Monthly data: {} rows x {} cols'.format(len(monthly), len(monthly.columns)))
train_q = q_monthly[(q_monthly['date'] >= TRAIN_START) & (q_monthly['date'] <= TRAIN_END)]
print('Training quarters: {} ({} to {})'.format(
    train_q[TARGET].notna().sum(),
    train_q['date'].min().strftime('%Y-%m'),
    train_q['date'].max().strftime('%Y-%m')
))


Monthly data: 628 rows x 59 cols
Training quarters: 67 (1995-03 to 2011-12)


In [3]:
# Rolling out-of-sample loop
# ARMA is univariate, but the available GDP history still depends on the
# simulated vintage date because the target has months_lag=2 in metadata.
# For each target quarter, refit ARMA(2,0,0) on the expanding history that
# would have been observable at m1/m2/m3.

test_quarters = monthly[
    (monthly['date'] >= TEST_START) &
    (monthly['date'] <= TEST_END) &
    monthly['date'].dt.month.isin([3, 6, 9, 12])
]['date'].tolist()

predictions  = {v: [] for v in VINTAGES}
actuals_list = []
failed = 0

for i, q_date in enumerate(test_quarters):
    pd_q = pd.Timestamp(q_date)
    actual_row = monthly[monthly['date'] == pd_q]
    actual = actual_row[TARGET].values[0] if len(actual_row) > 0 else np.nan
    actuals_list.append(actual)

    for vintage_name, offset in VINTAGES.items():
        forecast_date = pd_q + pd.DateOffset(months=offset)
        fc_monthly = monthly[
            (monthly['date'] >= TRAIN_START) &
            (monthly['date'] <= forecast_date)
        ][['date', TARGET]].reset_index(drop=True)

        masked = gen_lagged_data(metadata, fc_monthly.copy(), forecast_date, lag=0)
        history = masked[
            masked['date'].dt.month.isin([3, 6, 9, 12]) &
            (masked['date'] < pd_q) &
            masked[TARGET].notna()
        ][TARGET].astype(float).values

        try:
            if len(history) < 12:
                raise ValueError('Not enough GDP history for ARMA')
            model = ARIMA(order=(2, 0, 0))
            model.fit(history)
            pred = float(model.predict(n_periods=1)[0])
            predictions[vintage_name].append(pred)
        except Exception as exc:
            predictions[vintage_name].append(np.nan)
            failed += 1

    if (i + 1) % 8 == 0:
        print('  {} / {}'.format(i + 1, len(test_quarters)))

print('Done. {} test quarters. {} failures.'.format(len(test_quarters), failed))


  8 / 32


  16 / 32


  24 / 32


  32 / 32
Done. 32 test quarters. 0 failures.


In [4]:
os.makedirs(PREDICTIONS_DIR, exist_ok=True)

for vintage_name in VINTAGES:
    df = pd.DataFrame({
        'date':       test_quarters,
        'actual':     actuals_list,
        'prediction': predictions[vintage_name],
    })
    path = os.path.join(PREDICTIONS_DIR, 'arma_{}.csv'.format(vintage_name))
    df.to_csv(path, index=False)
    print('Saved {} ({} rows) pred=[{:.4f},{:.4f}]'.format(
        os.path.basename(path), len(df),
        df['prediction'].min(), df['prediction'].max()
    ))


Saved arma_m1.csv (32 rows) pred=[-0.0313,0.0222]
Saved arma_m2.csv (32 rows) pred=[-0.0313,0.0222]
Saved arma_m3.csv (32 rows) pred=[-0.0313,0.0222]
Saved arma_post1.csv (32 rows) pred=[-0.0313,0.0222]
Saved arma_post2.csv (32 rows) pred=[-0.0313,0.0222]


In [5]:
# Panel evaluation — Turkey economic periods
panels = {
    'Pre-crisis (2018-2019)': ('2018-01-01', '2019-12-31'),
    'COVID      (2020-2021)': ('2020-04-01', '2021-12-31'),
    'Post-COVID (2022-2025)': ('2022-01-01', '2025-12-31'),
    'Full test  (2018-2025)': ('2018-01-01', '2025-12-31'),
}

def rmse(a, p):
    mask = ~(np.isnan(a) | np.isnan(p))
    return np.sqrt(np.mean((a[mask] - p[mask]) ** 2)) if mask.sum() > 0 else np.nan

def mae(a, p):
    mask = ~(np.isnan(a) | np.isnan(p))
    return np.mean(np.abs(a[mask] - p[mask])) if mask.sum() > 0 else np.nan

actual_arr = np.array(actuals_list)
dates_arr  = pd.to_datetime(test_quarters)

print('=' * 72)
print('ARMA — Evaluation by Panel & Vintage (Turkey)')
print('=' * 72)

for panel_name, (p_start, p_end) in panels.items():
    print()
    print('--- {} ---'.format(panel_name))
    print('  Vintage    RMSFE      MAE       N')
    print('  -------    -----      ---       -')
    for vintage_name in VINTAGES:
        pred_arr = np.array(predictions[vintage_name])
        mask = (dates_arr >= p_start) & (dates_arr <= p_end)
        a = actual_arr[mask]
        p = pred_arr[mask]
        n_valid = (~(np.isnan(a) | np.isnan(p))).sum()
        print('  {:<10s} {:.6f}  {:.6f}  {}'.format(
            vintage_name, rmse(a, p), mae(a, p), n_valid))


ARMA — Evaluation by Panel & Vintage (Turkey)

--- Pre-crisis (2018-2019) ---
  Vintage    RMSFE      MAE       N
  -------    -----      ---       -
  m1         0.019041  0.014263  8
  m2         0.019041  0.014263  8
  m3         0.018588  0.015068  8
  post1      0.018588  0.015068  8
  post2      0.018588  0.015068  8

--- COVID      (2020-2021) ---
  Vintage    RMSFE      MAE       N
  -------    -----      ---       -
  m1         0.072537  0.052587  7
  m2         0.072537  0.052587  7
  m3         0.076027  0.051088  7
  post1      0.076027  0.051088  7
  post2      0.076027  0.051088  7

--- Post-COVID (2022-2025) ---
  Vintage    RMSFE      MAE       N
  -------    -----      ---       -
  m1         0.011679  0.007746  16
  m2         0.011679  0.007746  16
  m3         0.011574  0.007798  16
  post1      0.011574  0.007798  16
  post2      0.011574  0.007798  16

--- Full test  (2018-2025) ---
  Vintage    RMSFE      MAE       N
  -------    -----      ---       -
  m1    